# BiolytAI — Clinical Trial Data Cleaning & Harmonization
Fields: `intervention_name`, `condition`, `sponsor`, `country`

In [ ]:
!pip install pandas numpy rapidfuzz tqdm -q

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import unicodedata
from collections import Counter
from rapidfuzz import fuzz, process
from tqdm.auto import tqdm

tqdm.pandas()

df = pd.read_csv('trials_sample.csv')
print(f'Loaded {len(df)} rows x {len(df.columns)} cols')
df.head(3)

## Utility

In [ ]:
def normalize_text(s):
    """Lowercase, strip accents, collapse whitespace, remove punctuation noise."""
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFKD', s)
    s = s.encode('ascii', 'ignore').decode('ascii')
    s = s.lower().strip()
    s = re.sub(r'[\s]+', ' ', s)
    return s

def pick_canonical(variants, prefer_longest=True):
    """From a list of raw strings, pick the most informative one as canonical."""
    variants = [v for v in variants if isinstance(v, str) and v.strip()]
    if not variants:
        return ''
    # prefer title-case longest variant
    scored = sorted(variants, key=lambda x: (len(x), x[0].isupper()), reverse=True)
    return scored[0]

## 1. Country Harmonization
Countries are the most structured — use a known mapping table.

In [ ]:
COUNTRY_MAP = {
    # USA variants
    'usa': 'United States', 'us': 'United States', 'u.s.': 'United States',
    'u.s.a.': 'United States', 'united states of america': 'United States',
    'united states': 'United States', 'america': 'United States',
    # UK
    'uk': 'United Kingdom', 'u.k.': 'United Kingdom',
    'great britain': 'United Kingdom', 'england': 'United Kingdom',
    'scotland': 'United Kingdom', 'wales': 'United Kingdom',
    'britain': 'United Kingdom',
    # Common short forms
    'korea': 'South Korea', 'south korea': 'South Korea',
    'republic of korea': 'South Korea', 'dprk': 'North Korea',
    'taiwan': 'Taiwan', 'republic of china': 'Taiwan',
    'china': 'China', "people's republic of china": 'China',
    'prc': 'China',
    'iran': 'Iran', 'islamic republic of iran': 'Iran',
    'russia': 'Russia', 'russian federation': 'Russia',
    'czech republic': 'Czech Republic', 'czechia': 'Czech Republic',
    'slovak republic': 'Slovakia', 'slovakia': 'Slovakia',
    'turkey': 'Turkey', 'turkiye': 'Turkey',
    'uae': 'United Arab Emirates', 'u.a.e.': 'United Arab Emirates',
    'holland': 'Netherlands', 'the netherlands': 'Netherlands',
    'vietnam': 'Vietnam', 'viet nam': 'Vietnam',
    'congo': 'Congo',
    'democratic republic of the congo': 'Democratic Republic of the Congo',
    'drc': 'Democratic Republic of the Congo',
    'tanzania': 'Tanzania', 'united republic of tanzania': 'Tanzania',
    'bolivia': 'Bolivia', 'plurinational state of bolivia': 'Bolivia',
    'moldova': 'Moldova', 'republic of moldova': 'Moldova',
    'laos': 'Laos', "lao people's democratic republic": 'Laos',
    'myanmar': 'Myanmar', 'burma': 'Myanmar',
    'north macedonia': 'North Macedonia', 'macedonia': 'North Macedonia',
    'palestine': 'Palestine', 'west bank and gaza': 'Palestine',
    'hong kong': 'Hong Kong',
    'puerto rico': 'Puerto Rico',
}

def harmonize_country(raw):
    """Split pipe/comma-separated countries, normalize each, return sorted joined string."""
    if not isinstance(raw, str) or not raw.strip():
        return np.nan
    parts = re.split(r'[|,]', raw)
    result = []
    for p in parts:
        p = p.strip()
        key = normalize_text(p)
        canonical = COUNTRY_MAP.get(key, p.strip())  # fall back to title-cased original
        if canonical and canonical not in result:
            result.append(canonical)
    return ' | '.join(sorted(result)) if result else np.nan

before_country = df['country'].nunique()
df['country_clean'] = df['country'].progress_apply(harmonize_country)
after_country = df['country_clean'].nunique()
print(f'Country unique values: {before_country} → {after_country}')

## 2. Sponsor Harmonization
Strategy: normalize text → fuzzy cluster with RapidFuzz token_sort_ratio → pick longest as canonical.

In [ ]:
# Pre-built overrides for well-known sponsors where fuzzy alone isn't enough
SPONSOR_OVERRIDES = {
    # Pfizer
    'pfizer': 'Pfizer', 'pfizer inc': 'Pfizer', 'pfizer inc.': 'Pfizer',
    'pfizer, inc': 'Pfizer', 'pfizer, inc.': 'Pfizer',
    # Novartis
    'novartis': 'Novartis', 'novartis ag': 'Novartis',
    'novartis pharmaceuticals': 'Novartis',
    'novartis pharma ag': 'Novartis',
    # Roche / Genentech
    'roche': 'Roche', 'f. hoffmann-la roche': 'Roche',
    'f. hoffmann-la roche ltd': 'Roche', 'hoffmann-la roche': 'Roche',
    'genentech': 'Genentech / Roche', 'genentech, inc.': 'Genentech / Roche',
    # AstraZeneca
    'astrazeneca': 'AstraZeneca', 'astra zeneca': 'AstraZeneca',
    # MSD / Merck
    'merck': 'Merck', 'merck & co': 'Merck', 'merck & co.': 'Merck',
    'merck sharp & dohme': 'Merck', 'merck sharp & dohme llc': 'Merck',
    'msd': 'Merck', 'msd (oss) b.v.': 'Merck',
    # BMS
    'bristol-myers squibb': 'Bristol-Myers Squibb',
    'bristol myers squibb': 'Bristol-Myers Squibb',
    'bms': 'Bristol-Myers Squibb',
    # J&J / Janssen
    'johnson & johnson': 'Johnson & Johnson',
    'johnson and johnson': 'Johnson & Johnson',
    'janssen': 'Janssen (J&J)', 'janssen research & development': 'Janssen (J&J)',
    'janssen pharmaceutica': 'Janssen (J&J)',
    # Eli Lilly
    'eli lilly': 'Eli Lilly', 'eli lilly and company': 'Eli Lilly',
    'lilly': 'Eli Lilly',
    # AbbVie
    'abbvie': 'AbbVie', 'abbvie inc.': 'AbbVie', 'abbvie inc': 'AbbVie',
    # Sanofi
    'sanofi': 'Sanofi', 'sanofi aventis': 'Sanofi',
    'sanofi-aventis': 'Sanofi', 'sanofi-synthelabo': 'Sanofi',
    # GSK
    'glaxosmithkline': 'GlaxoSmithKline',
    'gsk': 'GlaxoSmithKline', 'smithkline beecham': 'GlaxoSmithKline',
    # Bayer
    'bayer': 'Bayer', 'bayer ag': 'Bayer', 'bayer healthcare': 'Bayer',
    'bayer healthcare pharmaceuticals': 'Bayer',
    # Boehringer
    'boehringer ingelheim': 'Boehringer Ingelheim',
    'boehringer ingelheim pharmaceuticals': 'Boehringer Ingelheim',
    # Amgen
    'amgen': 'Amgen', 'amgen inc.': 'Amgen', 'amgen inc': 'Amgen',
    # NIH / NCI
    'national cancer institute': 'National Cancer Institute (NCI)',
    'nci': 'National Cancer Institute (NCI)',
    'national institutes of health': 'National Institutes of Health (NIH)',
    'nih': 'National Institutes of Health (NIH)',
    # Takeda
    'takeda': 'Takeda', 'takeda pharmaceuticals': 'Takeda',
    'takeda pharmaceutical company': 'Takeda',
}

def clean_sponsor_key(s):
    if not isinstance(s, str):
        return ''
    s = normalize_text(s)
    s = re.sub(r'\b(llc|ltd|inc|corp|co|gmbh|ag|sa|plc|bv|nv|ab|oy|spa|sas|pte|pvt|limited|incorporated|company|pharmaceutical|pharmaceuticals|pharma)\b\.?', '', s)
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def harmonize_sponsor(raw):
    if not isinstance(raw, str) or not raw.strip():
        return raw
    key = normalize_text(raw)
    # Direct override
    if key in SPONSOR_OVERRIDES:
        return SPONSOR_OVERRIDES[key]
    # Partial match on cleaned key
    clean_key = clean_sponsor_key(raw)
    for pattern, canonical in SPONSOR_OVERRIDES.items():
        if clean_sponsor_key(pattern) and clean_sponsor_key(pattern) in clean_key:
            return canonical
    return raw.strip()

before_sponsor = df['sponsor'].nunique()
df['sponsor_clean'] = df['sponsor'].progress_apply(harmonize_sponsor)
after_sponsor = df['sponsor_clean'].nunique()
print(f'Sponsor unique values: {before_sponsor} → {after_sponsor}')

In [ ]:
# Fuzzy clustering for remaining sponsors (high-frequency ones)
# Build clusters of similar sponsor names (token_sort_ratio >= 88)

sponsor_counts = df['sponsor_clean'].value_counts()
# Only cluster sponsors appearing >= 2 times to keep runtime reasonable
sponsors_to_cluster = sponsor_counts[sponsor_counts >= 2].index.tolist()
print(f'Sponsors appearing >=2 times: {len(sponsors_to_cluster)}')

sponsor_cluster_map = {}  # variant -> canonical
visited = set()

for i, s in enumerate(tqdm(sponsors_to_cluster)):
    if s in visited:
        continue
    cluster = [s]
    visited.add(s)
    for j, t in enumerate(sponsors_to_cluster):
        if t in visited:
            continue
        score = fuzz.token_sort_ratio(clean_sponsor_key(s), clean_sponsor_key(t))
        if score >= 90:
            cluster.append(t)
            visited.add(t)
    canonical = pick_canonical(cluster)
    for v in cluster:
        sponsor_cluster_map[v] = canonical

df['sponsor_clean'] = df['sponsor_clean'].map(lambda x: sponsor_cluster_map.get(x, x))
after_sponsor_fuzzy = df['sponsor_clean'].nunique()
print(f'After fuzzy clustering: {after_sponsor_fuzzy} unique sponsors')

## 3. Drug / Intervention Harmonization
Strategy: split pipe-separated drugs → normalize → override map → RapidFuzz clustering on drug-specific cleaned keys.

In [ ]:
DRUG_OVERRIDES = {
    # PD-1/PD-L1
    'pembrolizumab': 'Pembrolizumab', 'pembro': 'Pembrolizumab',
    'keytruda': 'Pembrolizumab', 'mk-3475': 'Pembrolizumab',
    'mk3475': 'Pembrolizumab', 'lambrolizumab': 'Pembrolizumab',
    'nivolumab': 'Nivolumab', 'opdivo': 'Nivolumab', 'bms-936558': 'Nivolumab',
    'atezolizumab': 'Atezolizumab', 'tecentriq': 'Atezolizumab',
    'mpdl3280a': 'Atezolizumab',
    'durvalumab': 'Durvalumab', 'imfinzi': 'Durvalumab', 'medi4736': 'Durvalumab',
    'avelumab': 'Avelumab', 'bavencio': 'Avelumab',
    'ipilimumab': 'Ipilimumab', 'yervoy': 'Ipilimumab', 'mdi-010': 'Ipilimumab',
    # Targeted therapies
    'trastuzumab': 'Trastuzumab', 'herceptin': 'Trastuzumab',
    'pertuzumab': 'Pertuzumab', 'perjeta': 'Pertuzumab',
    'bevacizumab': 'Bevacizumab', 'avastin': 'Bevacizumab',
    'cetuximab': 'Cetuximab', 'erbitux': 'Cetuximab',
    'rituximab': 'Rituximab', 'rituxan': 'Rituximab', 'mabthera': 'Rituximab',
    'imatinib': 'Imatinib', 'gleevec': 'Imatinib', 'glivec': 'Imatinib',
    'erlotinib': 'Erlotinib', 'tarceva': 'Erlotinib',
    'gefitinib': 'Gefitinib', 'iressa': 'Gefitinib',
    'osimertinib': 'Osimertinib', 'tagrisso': 'Osimertinib', 'az9291': 'Osimertinib',
    'crizotinib': 'Crizotinib', 'xalkori': 'Crizotinib',
    'alectinib': 'Alectinib', 'alecensa': 'Alectinib',
    'vemurafenib': 'Vemurafenib', 'zelboraf': 'Vemurafenib',
    'dabrafenib': 'Dabrafenib', 'tafinlar': 'Dabrafenib',
    'trametinib': 'Trametinib', 'mekinist': 'Trametinib',
    'palbociclib': 'Palbociclib', 'ibrance': 'Palbociclib',
    'ribociclib': 'Ribociclib', 'kisqali': 'Ribociclib',
    'abemaciclib': 'Abemaciclib', 'verzenio': 'Abemaciclib',
    'olaparib': 'Olaparib', 'lynparza': 'Olaparib',
    'niraparib': 'Niraparib', 'zejula': 'Niraparib',
    'rucaparib': 'Rucaparib', 'rubraca': 'Rucaparib',
    'venetoclax': 'Venetoclax', 'venclexta': 'Venetoclax', 'abt-199': 'Venetoclax',
    'ibrutinib': 'Ibrutinib', 'imbruvica': 'Ibrutinib',
    'bortezomib': 'Bortezomib', 'velcade': 'Bortezomib',
    'carfilzomib': 'Carfilzomib', 'kyprolis': 'Carfilzomib',
    'lenalidomide': 'Lenalidomide', 'revlimid': 'Lenalidomide',
    'thalidomide': 'Thalidomide', 'thalomid': 'Thalidomide',
    'pomalidomide': 'Pomalidomide', 'pomalyst': 'Pomalidomide',
    'daratumumab': 'Daratumumab', 'darzalex': 'Daratumumab',
    'elotuzumab': 'Elotuzumab', 'empliciti': 'Elotuzumab',
    # Chemotherapy
    'paclitaxel': 'Paclitaxel', 'taxol': 'Paclitaxel',
    'docetaxel': 'Docetaxel', 'taxotere': 'Docetaxel',
    'cisplatin': 'Cisplatin', 'platinol': 'Cisplatin',
    'carboplatin': 'Carboplatin', 'paraplatin': 'Carboplatin',
    'oxaliplatin': 'Oxaliplatin', 'eloxatin': 'Oxaliplatin',
    'gemcitabine': 'Gemcitabine', 'gemzar': 'Gemcitabine',
    'doxorubicin': 'Doxorubicin', 'adriamycin': 'Doxorubicin',
    'cyclophosphamide': 'Cyclophosphamide', 'cytoxan': 'Cyclophosphamide',
    'fluorouracil': 'Fluorouracil', '5-fluorouracil': 'Fluorouracil',
    '5-fu': 'Fluorouracil', '5fu': 'Fluorouracil',
    'capecitabine': 'Capecitabine', 'xeloda': 'Capecitabine',
    'irinotecan': 'Irinotecan', 'camptosar': 'Irinotecan',
    'vinorelbine': 'Vinorelbine', 'navelbine': 'Vinorelbine',
    'etoposide': 'Etoposide', 'vepesid': 'Etoposide',
    'pemetrexed': 'Pemetrexed', 'alimta': 'Pemetrexed',
    'methotrexate': 'Methotrexate', 'mtx': 'Methotrexate',
    # Hormonal
    'tamoxifen': 'Tamoxifen', 'nolvadex': 'Tamoxifen',
    'letrozole': 'Letrozole', 'femara': 'Letrozole',
    'anastrozole': 'Anastrozole', 'arimidex': 'Anastrozole',
    'exemestane': 'Exemestane', 'aromasin': 'Exemestane',
    'enzalutamide': 'Enzalutamide', 'xtandi': 'Enzalutamide',
    'abiraterone': 'Abiraterone', 'zytiga': 'Abiraterone',
    'leuprolide': 'Leuprolide', 'lupron': 'Leuprolide',
    # Antidiabetics
    'metformin': 'Metformin', 'glucophage': 'Metformin',
    'insulin': 'Insulin', 'insulin glargine': 'Insulin Glargine', 'lantus': 'Insulin Glargine',
    'semaglutide': 'Semaglutide', 'ozempic': 'Semaglutide', 'wegovy': 'Semaglutide',
    'liraglutide': 'Liraglutide', 'victoza': 'Liraglutide',
    'empagliflozin': 'Empagliflozin', 'jardiance': 'Empagliflozin',
    'dapagliflozin': 'Dapagliflozin', 'farxiga': 'Dapagliflozin', 'forxiga': 'Dapagliflozin',
    # Anticoagulants
    'warfarin': 'Warfarin', 'coumadin': 'Warfarin',
    'apixaban': 'Apixaban', 'eliquis': 'Apixaban',
    'rivaroxaban': 'Rivaroxaban', 'xarelto': 'Rivaroxaban',
    'dabigatran': 'Dabigatran', 'pradaxa': 'Dabigatran',
    # Statins
    'atorvastatin': 'Atorvastatin', 'lipitor': 'Atorvastatin',
    'rosuvastatin': 'Rosuvastatin', 'crestor': 'Rosuvastatin',
    'simvastatin': 'Simvastatin', 'zocor': 'Simvastatin',
    # Steroids
    'dexamethasone': 'Dexamethasone', 'decadron': 'Dexamethasone',
    'prednisone': 'Prednisone', 'prednisolone': 'Prednisolone',
    'methylprednisolone': 'Methylprednisolone', 'medrol': 'Methylprednisolone',
    # Antihypertensives
    'lisinopril': 'Lisinopril', 'zestril': 'Lisinopril', 'prinivil': 'Lisinopril',
    'amlodipine': 'Amlodipine', 'norvasc': 'Amlodipine',
    'losartan': 'Losartan', 'cozaar': 'Losartan',
    # Vaccines
    'placebo': 'Placebo',
    'standard of care': 'Standard of Care', 'standard care': 'Standard of Care',
    'best supportive care': 'Best Supportive Care', 'bsc': 'Best Supportive Care',
    'usual care': 'Usual Care',
}

def clean_drug_key(s):
    if not isinstance(s, str):
        return ''
    s = normalize_text(s)
    # Remove common suffixes that cloud matching
    s = re.sub(r'\b(hydrochloride|hcl|mesylate|acetate|sulfate|phosphate|sodium|potassium|injection|tablet|capsule|solution|oral|iv|sc|im|topical)\b', '', s)
    s = re.sub(r'[^a-z0-9 \-]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def harmonize_single_drug(raw_drug):
    raw_drug = raw_drug.strip()
    key = normalize_text(raw_drug)
    if key in DRUG_OVERRIDES:
        return DRUG_OVERRIDES[key]
    # Partial / contained match
    clean = clean_drug_key(raw_drug)
    for pattern, canonical in DRUG_OVERRIDES.items():
        p_clean = clean_drug_key(pattern)
        if p_clean and len(p_clean) >= 4 and p_clean in clean:
            return canonical
    return raw_drug

def harmonize_drug(raw):
    """Handle pipe-separated multi-drug strings."""
    if not isinstance(raw, str) or not raw.strip():
        return raw
    parts = [p.strip() for p in raw.split('|') if p.strip()]
    harmonized = [harmonize_single_drug(p) for p in parts]
    return ' | '.join(harmonized)

before_drug = df['intervention_name'].nunique()
df['intervention_clean'] = df['intervention_name'].progress_apply(harmonize_drug)
after_drug = df['intervention_clean'].nunique()
print(f'Intervention unique values: {before_drug} → {after_drug}')

In [ ]:
# Fuzzy clustering on single-drug entries (no pipe) that appear >= 3 times
single_drug_mask = ~df['intervention_clean'].str.contains('|', regex=False, na=True)
single_drug_counts = df.loc[single_drug_mask, 'intervention_clean'].value_counts()
drugs_to_cluster = single_drug_counts[single_drug_counts >= 3].index.tolist()
print(f'Single-drug entries to cluster: {len(drugs_to_cluster)}')

drug_cluster_map = {}
visited = set()

for s in tqdm(drugs_to_cluster):
    if s in visited:
        continue
    cluster = [s]
    visited.add(s)
    for t in drugs_to_cluster:
        if t in visited:
            continue
        if abs(len(s) - len(t)) > 20:  # skip obviously different lengths
            continue
        score = fuzz.token_sort_ratio(clean_drug_key(s), clean_drug_key(t))
        if score >= 88:
            cluster.append(t)
            visited.add(t)
    canonical = pick_canonical(cluster)
    for v in cluster:
        drug_cluster_map[v] = canonical

def apply_drug_cluster(raw):
    if not isinstance(raw, str):
        return raw
    if '|' in raw:
        parts = [drug_cluster_map.get(p.strip(), p.strip()) for p in raw.split('|')]
        return ' | '.join(parts)
    return drug_cluster_map.get(raw, raw)

df['intervention_clean'] = df['intervention_clean'].apply(apply_drug_cluster)
print(f'After fuzzy clustering: {df["intervention_clean"].nunique()} unique interventions')

## 4. Disease / Condition Harmonization
Conditions can be multi-label (pipe-separated). Strategy: per-label override map + fuzzy clustering.

In [ ]:
DISEASE_OVERRIDES = {
    # Lung cancers
    'non-small cell lung cancer': 'Non-Small Cell Lung Cancer (NSCLC)',
    'non-small-cell lung cancer': 'Non-Small Cell Lung Cancer (NSCLC)',
    'nsclc': 'Non-Small Cell Lung Cancer (NSCLC)',
    'non small cell lung cancer': 'Non-Small Cell Lung Cancer (NSCLC)',
    'nonsmall cell lung cancer': 'Non-Small Cell Lung Cancer (NSCLC)',
    'small cell lung cancer': 'Small Cell Lung Cancer (SCLC)',
    'sclc': 'Small Cell Lung Cancer (SCLC)',
    'lung cancer': 'Lung Cancer',
    'lung cancers': 'Lung Cancer',
    'lung neoplasm': 'Lung Cancer',
    'lung neoplasms': 'Lung Cancer',
    'carcinoma, non-small-cell lung': 'Non-Small Cell Lung Cancer (NSCLC)',
    'carcinoma, non-small cell lung': 'Non-Small Cell Lung Cancer (NSCLC)',
    # Breast cancer
    'breast cancer': 'Breast Cancer',
    'breast cancers': 'Breast Cancer',
    'breast neoplasm': 'Breast Cancer',
    'breast neoplasms': 'Breast Cancer',
    'breast carcinoma': 'Breast Cancer',
    'carcinoma, breast': 'Breast Cancer',
    'her2-positive breast cancer': 'HER2-Positive Breast Cancer',
    'her2+ breast cancer': 'HER2-Positive Breast Cancer',
    'triple-negative breast cancer': 'Triple-Negative Breast Cancer (TNBC)',
    'triple negative breast cancer': 'Triple-Negative Breast Cancer (TNBC)',
    'tnbc': 'Triple-Negative Breast Cancer (TNBC)',
    # Colorectal
    'colorectal cancer': 'Colorectal Cancer',
    'colorectal carcinoma': 'Colorectal Cancer',
    'colon cancer': 'Colon Cancer',
    'rectal cancer': 'Rectal Cancer',
    'colon carcinoma': 'Colon Cancer',
    'carcinoma, colorectal': 'Colorectal Cancer',
    # Prostate
    'prostate cancer': 'Prostate Cancer',
    'prostate carcinoma': 'Prostate Cancer',
    'carcinoma, prostate': 'Prostate Cancer',
    'prostatic neoplasm': 'Prostate Cancer',
    'castration-resistant prostate cancer': 'Castration-Resistant Prostate Cancer (CRPC)',
    'crpc': 'Castration-Resistant Prostate Cancer (CRPC)',
    # Ovarian
    'ovarian cancer': 'Ovarian Cancer',
    'ovarian carcinoma': 'Ovarian Cancer',
    'carcinoma, ovarian': 'Ovarian Cancer',
    'ovarian neoplasm': 'Ovarian Cancer',
    # Melanoma
    'melanoma': 'Melanoma',
    'malignant melanoma': 'Melanoma',
    'cutaneous melanoma': 'Melanoma',
    'skin melanoma': 'Melanoma',
    # Pancreatic
    'pancreatic cancer': 'Pancreatic Cancer',
    'pancreatic carcinoma': 'Pancreatic Cancer',
    'carcinoma, pancreatic': 'Pancreatic Cancer',
    'adenocarcinoma, pancreatic ductal': 'Pancreatic Ductal Adenocarcinoma (PDAC)',
    'pdac': 'Pancreatic Ductal Adenocarcinoma (PDAC)',
    # Liver
    'hepatocellular carcinoma': 'Hepatocellular Carcinoma (HCC)',
    'hcc': 'Hepatocellular Carcinoma (HCC)',
    'carcinoma, hepatocellular': 'Hepatocellular Carcinoma (HCC)',
    'liver cancer': 'Hepatocellular Carcinoma (HCC)',
    'liver cell carcinoma': 'Hepatocellular Carcinoma (HCC)',
    # Leukemia
    'acute myeloid leukemia': 'Acute Myeloid Leukemia (AML)',
    'aml': 'Acute Myeloid Leukemia (AML)',
    'acute lymphoblastic leukemia': 'Acute Lymphoblastic Leukemia (ALL)',
    'all': 'Acute Lymphoblastic Leukemia (ALL)',
    'chronic myeloid leukemia': 'Chronic Myeloid Leukemia (CML)',
    'cml': 'Chronic Myeloid Leukemia (CML)',
    'chronic lymphocytic leukemia': 'Chronic Lymphocytic Leukemia (CLL)',
    'cll': 'Chronic Lymphocytic Leukemia (CLL)',
    'leukemia': 'Leukemia',
    # Lymphoma
    'diffuse large b-cell lymphoma': 'Diffuse Large B-Cell Lymphoma (DLBCL)',
    'dlbcl': 'Diffuse Large B-Cell Lymphoma (DLBCL)',
    'hodgkin lymphoma': 'Hodgkin Lymphoma',
    "hodgkin's lymphoma": 'Hodgkin Lymphoma',
    "hodgkin's disease": 'Hodgkin Lymphoma',
    'non-hodgkin lymphoma': 'Non-Hodgkin Lymphoma',
    'non hodgkin lymphoma': 'Non-Hodgkin Lymphoma',
    'nhl': 'Non-Hodgkin Lymphoma',
    'lymphoma': 'Lymphoma',
    # Myeloma
    'multiple myeloma': 'Multiple Myeloma',
    'myeloma': 'Multiple Myeloma',
    # Cardiovascular
    'heart failure': 'Heart Failure',
    'cardiac failure': 'Heart Failure',
    'congestive heart failure': 'Heart Failure',
    'chf': 'Heart Failure',
    'myocardial infarction': 'Myocardial Infarction',
    'heart attack': 'Myocardial Infarction',
    'acute myocardial infarction': 'Myocardial Infarction',
    'coronary artery disease': 'Coronary Artery Disease',
    'cad': 'Coronary Artery Disease',
    'hypertension': 'Hypertension',
    'high blood pressure': 'Hypertension',
    'arterial hypertension': 'Hypertension',
    # Diabetes
    'type 2 diabetes': 'Type 2 Diabetes Mellitus',
    'type 2 diabetes mellitus': 'Type 2 Diabetes Mellitus',
    't2dm': 'Type 2 Diabetes Mellitus',
    'type 1 diabetes': 'Type 1 Diabetes Mellitus',
    'type 1 diabetes mellitus': 'Type 1 Diabetes Mellitus',
    't1dm': 'Type 1 Diabetes Mellitus',
    'diabetes': 'Diabetes Mellitus',
    'diabetes mellitus': 'Diabetes Mellitus',
    # Neurological
    "alzheimer's disease": "Alzheimer's Disease",
    'alzheimer disease': "Alzheimer's Disease",
    'alzheimers disease': "Alzheimer's Disease",
    "parkinson's disease": "Parkinson's Disease",
    'parkinson disease': "Parkinson's Disease",
    'parkinsons disease': "Parkinson's Disease",
    'multiple sclerosis': 'Multiple Sclerosis',
    'ms': 'Multiple Sclerosis',
    'epilepsy': 'Epilepsy',
    'stroke': 'Stroke',
    'ischemic stroke': 'Ischemic Stroke',
    'cerebrovascular accident': 'Stroke',
    # Autoimmune
    'rheumatoid arthritis': 'Rheumatoid Arthritis',
    'ra': 'Rheumatoid Arthritis',
    'systemic lupus erythematosus': 'Systemic Lupus Erythematosus (SLE)',
    'lupus': 'Systemic Lupus Erythematosus (SLE)',
    'sle': 'Systemic Lupus Erythematosus (SLE)',
    "crohn's disease": "Crohn's Disease",
    'crohn disease': "Crohn's Disease",
    'ulcerative colitis': 'Ulcerative Colitis',
    # Respiratory
    'chronic obstructive pulmonary disease': 'COPD',
    'copd': 'COPD',
    'asthma': 'Asthma',
    'pulmonary fibrosis': 'Pulmonary Fibrosis',
    'idiopathic pulmonary fibrosis': 'Idiopathic Pulmonary Fibrosis (IPF)',
    'ipf': 'Idiopathic Pulmonary Fibrosis (IPF)',
    # Infectious
    'hiv': 'HIV/AIDS',
    'hiv infection': 'HIV/AIDS',
    'hiv infections': 'HIV/AIDS',
    'aids': 'HIV/AIDS',
    'covid-19': 'COVID-19',
    'covid19': 'COVID-19',
    'coronavirus': 'COVID-19',
    'sars-cov-2': 'COVID-19',
    'hepatitis b': 'Hepatitis B',
    'hepatitis c': 'Hepatitis C',
    'tuberculosis': 'Tuberculosis',
    'tb': 'Tuberculosis',
    # Kidney
    'renal cell carcinoma': 'Renal Cell Carcinoma (RCC)',
    'rcc': 'Renal Cell Carcinoma (RCC)',
    'kidney cancer': 'Renal Cell Carcinoma (RCC)',
    'chronic kidney disease': 'Chronic Kidney Disease (CKD)',
    'ckd': 'Chronic Kidney Disease (CKD)',
    'renal failure': 'Renal Failure',
    # Bladder
    'bladder cancer': 'Bladder Cancer',
    'urothelial carcinoma': 'Urothelial Carcinoma',
    # Glioma
    'glioblastoma': 'Glioblastoma (GBM)',
    'gbm': 'Glioblastoma (GBM)',
    'glioblastoma multiforme': 'Glioblastoma (GBM)',
    'glioma': 'Glioma',
    # Head/neck
    'head and neck cancer': 'Head and Neck Cancer',
    'head and neck squamous cell carcinoma': 'Head and Neck Squamous Cell Carcinoma (HNSCC)',
    'hnscc': 'Head and Neck Squamous Cell Carcinoma (HNSCC)',
    # Gastric
    'gastric cancer': 'Gastric Cancer',
    'stomach cancer': 'Gastric Cancer',
    'gastric carcinoma': 'Gastric Cancer',
    # Endometrial
    'endometrial cancer': 'Endometrial Cancer',
    'uterine cancer': 'Endometrial Cancer',
    # Sarcoma
    'sarcoma': 'Sarcoma',
    'soft tissue sarcoma': 'Soft Tissue Sarcoma',
    # Thyroid
    'thyroid cancer': 'Thyroid Cancer',
    'papillary thyroid carcinoma': 'Papillary Thyroid Cancer',
}

def harmonize_single_condition(raw_cond):
    raw_cond = raw_cond.strip()
    key = normalize_text(raw_cond)
    if key in DISEASE_OVERRIDES:
        return DISEASE_OVERRIDES[key]
    # Partial match on first token
    for pattern, canonical in DISEASE_OVERRIDES.items():
        if len(pattern) >= 5 and pattern in key:
            return canonical
    return raw_cond

def harmonize_condition(raw):
    if not isinstance(raw, str) or not raw.strip():
        return raw
    parts = [p.strip() for p in raw.split('|') if p.strip()]
    harmonized = [harmonize_single_condition(p) for p in parts]
    # Deduplicate while preserving order
    seen = set()
    deduped = []
    for h in harmonized:
        if h not in seen:
            seen.add(h)
            deduped.append(h)
    return ' | '.join(deduped)

before_disease = df['condition'].nunique()
df['condition_clean'] = df['condition'].progress_apply(harmonize_condition)
after_disease = df['condition_clean'].nunique()
print(f'Condition unique values: {before_disease} → {after_disease}')

In [ ]:
# Fuzzy cluster single-condition entries appearing >= 3 times
single_cond_mask = ~df['condition_clean'].str.contains('|', regex=False, na=True)
single_cond_counts = df.loc[single_cond_mask, 'condition_clean'].value_counts()
conds_to_cluster = single_cond_counts[single_cond_counts >= 3].index.tolist()
print(f'Single-condition entries to cluster: {len(conds_to_cluster)}')

disease_cluster_map = {}
visited = set()

def clean_disease_key(s):
    if not isinstance(s, str):
        return ''
    s = normalize_text(s)
    s = re.sub(r'\b(disease|disorder|syndrome|condition|cancer|carcinoma|neoplasm)s?\b', '', s)
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

for s in tqdm(conds_to_cluster):
    if s in visited:
        continue
    cluster = [s]
    visited.add(s)
    for t in conds_to_cluster:
        if t in visited:
            continue
        if abs(len(s) - len(t)) > 25:
            continue
        score = fuzz.token_sort_ratio(clean_disease_key(s), clean_disease_key(t))
        if score >= 88:
            cluster.append(t)
            visited.add(t)
    canonical = pick_canonical(cluster)
    for v in cluster:
        disease_cluster_map[v] = canonical

def apply_disease_cluster(raw):
    if not isinstance(raw, str):
        return raw
    if '|' in raw:
        parts = [disease_cluster_map.get(p.strip(), p.strip()) for p in raw.split('|')]
        return ' | '.join(dict.fromkeys(parts))  # dedup
    return disease_cluster_map.get(raw, raw)

df['condition_clean'] = df['condition_clean'].apply(apply_disease_cluster)
print(f'After fuzzy clustering: {df["condition_clean"].nunique()} unique conditions')

## 5. Build Dictionaries

In [ ]:
def build_dictionary(df, raw_col, clean_col, sep='|'):
    """Expand multi-value pipe-separated fields and build variant->canonical dict."""
    mapping = {}
    for raw, clean in zip(df[raw_col], df[clean_col]):
        if not isinstance(raw, str) or not isinstance(clean, str):
            continue
        raw_parts = [p.strip() for p in raw.split(sep)] if sep else [raw.strip()]
        clean_parts = [p.strip() for p in clean.split(sep)] if sep else [clean.strip()]
        # Align by index (best effort)
        for i, rp in enumerate(raw_parts):
            cp = clean_parts[i] if i < len(clean_parts) else clean_parts[-1]
            if rp and cp and rp != cp:
                mapping[rp] = cp
    return mapping

drug_dict = build_dictionary(df, 'intervention_name', 'intervention_clean', sep='|')
disease_dict = build_dictionary(df, 'condition', 'condition_clean', sep='|')
sponsor_dict = build_dictionary(df, 'sponsor', 'sponsor_clean', sep=None)
country_dict = build_dictionary(df, 'country', 'country_clean', sep='|')

print(f'Drug dictionary: {len(drug_dict)} mappings')
print(f'Disease dictionary: {len(disease_dict)} mappings')
print(f'Sponsor dictionary: {len(sponsor_dict)} mappings')
print(f'Country dictionary: {len(country_dict)} mappings')

In [ ]:
# Save dictionaries
import os
os.makedirs('output', exist_ok=True)

with open('output/drug_dictionary.json', 'w') as f:
    json.dump(drug_dict, f, indent=2, ensure_ascii=False)

with open('output/disease_dictionary.json', 'w') as f:
    json.dump(disease_dict, f, indent=2, ensure_ascii=False)

with open('output/sponsor_dictionary.json', 'w') as f:
    json.dump(sponsor_dict, f, indent=2, ensure_ascii=False)

with open('output/country_dictionary.json', 'w') as f:
    json.dump(country_dict, f, indent=2, ensure_ascii=False)

print('Dictionaries saved.')

## 6. Quality Metrics

In [ ]:
metrics = {
    'intervention_name': {
        'unique_before': int(df['intervention_name'].nunique()),
        'unique_after': int(df['intervention_clean'].nunique()),
        'reduction_pct': round((1 - df['intervention_clean'].nunique() / df['intervention_name'].nunique()) * 100, 1)
    },
    'condition': {
        'unique_before': int(df['condition'].nunique()),
        'unique_after': int(df['condition_clean'].nunique()),
        'reduction_pct': round((1 - df['condition_clean'].nunique() / df['condition'].nunique()) * 100, 1)
    },
    'sponsor': {
        'unique_before': int(df['sponsor'].nunique()),
        'unique_after': int(df['sponsor_clean'].nunique()),
        'reduction_pct': round((1 - df['sponsor_clean'].nunique() / df['sponsor'].nunique()) * 100, 1)
    },
    'country': {
        'unique_before': int(df['country'].nunique()),
        'unique_after': int(df['country_clean'].nunique()),
        'reduction_pct': round((1 - df['country_clean'].nunique() / df['country'].nunique()) * 100, 1)
    },
}

metrics_df = pd.DataFrame(metrics).T
print(metrics_df.to_string())

with open('output/harmonization_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

## 7. Stretch Goal — Before vs After: Pembrolizumab & Lung Cancer

In [ ]:
# Pembrolizumab
pembro_terms = ['Pembrolizumab', 'pembrolizumab', 'pembro', 'KEYTRUDA', 'Keytruda', 'MK-3475', 'mk-3475', 'lambrolizumab']
pembro_before = df['intervention_name'].str.contains('|'.join(re.escape(t) for t in pembro_terms), case=False, na=False).sum()
pembro_after = df['intervention_clean'].str.contains('Pembrolizumab', case=False, na=False).sum()
print(f'Pembrolizumab trials — before: {pembro_before}, after: {pembro_after}')

# Lung cancer
lung_terms_before = ['lung cancer', 'lung cancers', 'nsclc', 'non-small cell lung', 'non small cell lung', 'small cell lung', 'lung neoplasm']
lung_before = df['condition'].str.contains('|'.join(re.escape(t) for t in lung_terms_before), case=False, na=False).sum()
lung_after = df['condition_clean'].str.contains('lung cancer|NSCLC|SCLC', case=False, na=False).sum()
print(f'Lung cancer trials — before: {lung_before}, after: {lung_after}')

print()
print('Why the counts increase after harmonization:')
print('Before: a search for "lung cancer" misses trials labelled "NSCLC", "Non-small cell lung cancer", "lung neoplasm" etc.')
print('After: all those variants map to a canonical label, so one query surfaces them all.')

## 8. Save Harmonized CSV (intermediate — biomarkers will be added in notebook 2)

In [ ]:
# Rename clean columns for output
df_out = df.copy()
df_out = df_out.rename(columns={
    'intervention_clean': 'intervention_name_clean',
    'condition_clean': 'condition_clean',
    'sponsor_clean': 'sponsor_clean',
    'country_clean': 'country_clean',
})

df_out.to_csv('output/trials_harmonized_step1.csv', index=False)
print(f'Saved {len(df_out)} rows to output/trials_harmonized_step1.csv')
print('Columns:', [c for c in df_out.columns])